<a href="https://colab.research.google.com/github/vcellmike/PatternsFormation/blob/main/Working/2024_08_21_XGBoost_on_ImageJ_Features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json
import os
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from sklearn.metrics import accuracy_score, classification_report
from transformers import AutoModel, AutoTokenizer, get_scheduler
from torch.utils.data import Dataset, DataLoader, RandomSampler, SequentialSampler
from torch.optim import AdamW
from tqdm.notebook import tqdm, trange
from time import perf_counter
from PIL import Image
import pandas as pd
#from google.colab import drive
from PIL import Image
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import pickle

print("All dependencies present.")

In [ ]:
# set random seeds for repeatability
import numpy as np
import random

def set_seed(seed_val):
    random.seed(seed_val)
    np.random.seed(seed_val)
    torch.manual_seed(seed_val)
    torch.cuda.manual_seed_all(seed_val)

print("Seed set")

In [ ]:
seed_val = 42
set_seed(seed_val)

In [ ]:
# !unzip "/content/drive/Shareddrives/Summer_2023_VCell_Ml/2024-05-09 New Work/Saved_variables/05-09-2024_all_images-20240509T194653Z-001.zip"
# !unzip "/content/drive/Shareddrives/Summer_2023_VCell_Ml/2024-05-09 New Work/New Image Generation/Generated Images-20240523T233205Z-001.zip"
# !unzip "/content/drive/Shareddrives/Summer_2023_VCell_Ml/2024-05-09 New Work/New Image Generation/Generated_Images_7-2-24-20240702T205017Z-001.zip"
# !unzip "/content/drive/MyDrive/2024_Summer_Work_(Shared_Drive_Is_Full)/As of 2024-08-22 Saved variables and data/Real_Images/2024-08-22_ImageJ_Data/Regular.zip"
# !unzip "/content/drive/MyDrive/2024_Summer_Work_(Shared_Drive_Is_Full)/As of 2024-08-22 Saved variables and data/Real_Images/2024-08-22_ImageJ_Data/Inverse.zip"
!powershell -Command "Expand-Archive -Path './content/ImageJ_data/Inverse_resized.zip' -DestinationPath './content/ImageJ_data'" 

#!unzip "/content/drive/MyDrive/2024_Summer_Work_(Shared_Drive_Is_Full)/As of 2024-08-22 Saved variables and data/Real_Images/2024-08-22_ImageJ_Data/Regular_resized.zip"

!powershell -Command "Expand-Archive -Path './content/ImageJ_data/Regular_resized.zip' -DestinationPath './content/ImageJ_data'" 

#!unzip "/content/drive/MyDrive/2024_Summer_Work_(Shared_Drive_Is_Full)/As of 2024-08-22 Saved variables and data/Real_Images/2024-08-22_ImageJ_Data/Inverse_resized.zip"
!echo "Unzip Complete"

In [ ]:
import numpy
numpy.__version__

In [ ]:
with open("./data/data_old/2024-08-18_curr_df_np126.pkl", 'rb') as f:
  feats_df = pickle.load(f) # deserialize using load()

feats_df.head()
#feats_df = feats_df.drop([700,1159,4061], axis = "rows")
#feats_df.reset_index(inplace = True, drop = True)
#feats_df["class"] = np.array(feats_df["class"].astype(int) -1).astype(int)
#print(np.unique(np.array(feats_df["class"]), return_counts = True))

In [ ]:
with open("./data/data_old/2024-08-19_feat_array_unscaled.pkl", 'rb') as f:
  feat_array_classified = pickle.load(f) # deserialize using load()

with open("./data/data_old/2024-08-18_curr_df_np126.pkl", 'rb') as f:
  df_classes_classified = pickle.load(f) # deserialize using load()

In [ ]:
import os
os.getcwd()

In [ ]:
with open("./data/data_old/2024-08-19_feats_df_narrow_np126.pkl", 'rb') as f:
  feats_df_new = pickle.load(f) # deserialize using load()

feats_df.shape

In [ ]:
df_classes_classified.head()

In [ ]:
feats_df.head()

In [ ]:
# Sets pred-class column equal to class column
feats_df["pred_class"] = feats_df["class"]

In [ ]:
# two dataframes 
df1 = df_classes_classified[["noise","path","seed","dir","pred_class"]]
df2 = feats_df[["noise","path","seed","dir","pred_class"]]

# concat's predicted and manual classified dataframes (df_classes_classified + feats_df)
pred_and_manual_df = pd.concat([df1, df2], axis=0)

pred_and_manual_df.reset_index(inplace = True, drop = True)
feats_df_new.reset_index(inplace = True, drop = True)

In [ ]:
#add full path to each dataframe to allow matching based on unique path
# df with all params: feats_df_new
# df with predicted classes: pred_and_manual_df

fdf_full_path = []
for i in range(feats_df_new.shape[0]):
  fdf_full_path.append(feats_df_new["dir"][i] + feats_df_new["path"][i])


cdf_full_path = []
for j in range(pred_and_manual_df.shape[0]):
  cdf_full_path.append((pred_and_manual_df["dir"][j] + pred_and_manual_df["path"][j]))


feats_df_new["full_path"] = fdf_full_path
pred_and_manual_df["full_path"] = cdf_full_path

In [ ]:
indices_1 = []
indices_2 = []
errors = 0
counter = 0
for i in range(feats_df_new.shape[0]):
  #pull up path to search for
  path = pred_and_manual_df["full_path"][i]

  #find index within feats_df that matches path
  ind_df = feats_df_new[feats_df_new["full_path"] == path]

  if ind_df.shape[0] == 1:
    ind = ind_df.index[0]
    # print(ind_df.head())
    indices_1.append(ind)
    indices_2.append(i)
  else:
    # indices.append("error")
    # print("error")
    errors += 1


  if counter % 1000 == 0:
    print(counter)

  counter += 1


print("Complete.")

In [ ]:
# match dfs

feats_df_new = feats_df_new.iloc[indices_1]
pred_and_manual_df = pred_and_manual_df.iloc[indices_2]
feats_df_new.reset_index(inplace = True, drop = True)
pred_and_manual_df.reset_index(inplace = True, drop = True)

In [ ]:
same = list(np.array(feats_df_new["full_path"]) == np.array(pred_and_manual_df["full_path"]))

print(same.count(True))
print(same.count(False))
print(len(same))

In [ ]:
feats_df_new["classifier_pred_class"] = pred_and_manual_df["pred_class"]

In [ ]:
#its a feats_df of constrained parameters
feats_df_new.head()

feats_df = feats_df_new

In [ ]:
feats_df.head()

In [ ]:
print(feats_df.columns[13:111])

XGBOOST

In [ ]:
#! pip install xgboost
#! pip install openpyxl
#! pip install tensorflow
! pip install graphviz
! pip install hyperopt

from sklearn.multioutput import MultiOutputRegressor
from sklearn.svm import SVR
import numpy as np
from sklearn.model_selection import RepeatedKFold
from numpy import absolute
from pandas import read_csv
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import RepeatedKFold
from xgboost import XGBRegressor
import openpyxl
from xgboost import cv
from PIL import Image
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import operator
# for loading/processing the images
import tensorflow
from tensorflow.keras.utils import load_img
from tensorflow.keras.utils import img_to_array
from keras.applications.vgg16 import preprocess_input

# models
from keras.applications.vgg16 import VGG16
from keras.models import Model

# clustering and dimension reduction
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import pickle
from sklearn.ensemble import RandomForestRegressor
from sklearn import tree
import graphviz
from sklearn import metrics
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV
import xgboost as xgb
from hyperopt import fmin, tpe, hp,STATUS_OK
from sklearn.model_selection import KFold, cross_val_score
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score

print("All dependencies present")

In [ ]:
np.unique(feats_df["num_spots"].astype(float))

In [ ]:
# scale data

## X (independent variable) --> the feature values themselves
X = feats_df[feats_df.columns[13:111]].astype(float)

## Y (dependent variables) --> the parameter values of Negan and RTO themselves
y = feats_df[["Ua","Ui","Ga","Gi","Da","Di","Ba"]].astype(float)

#Scale data with standardscaler
scaling=StandardScaler()

# Use fit and transform method
scaling.fit(X)
X_scaled = scaling.transform(X)

# select 20 percent for testing
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42) #create training split



# df_train = feats_df.sample(frac = 0.80, random_state = 42)
# df_train.reset_index(inplace = True, drop = True)
# df_val_test = feats_df.drop(df_train.index)

# p_out = 1

# df_val_test = df_val_test.sample(frac = p_out, random_state = 42)
# df_val = df_val_test.sample(frac = 0.5, random_state = 42)
# df_test = df_val_test.drop(df_val.index)

# df_val.reset_index(inplace = True, drop = True)
# df_test.reset_index(inplace = True, drop = True)

In [ ]:
y_test

In [ ]:
model = xgb.XGBRegressor(n_estimators=1000, max_depth=10, eta=0.1, subsample=0.7, colsample_bytree=0.8, num_boost_round=50, objective= "reg:squarederror", device = "cuda")
model.fit(X_train, y_train)

In [ ]:
# make predictions
y_pred = model.predict(X_test)

In [ ]:
# GOAL: To determine the error rate in predicting the value of continuous, parameter data

# Converts test Y values to numpy array
true_vals = np.array(y_test)

# Outputs amount of true values
print(true_vals.shape)

# Outputs amount of predicted values
print(y_pred.shape)

# Sets counters for correct, incorrect, errors numpy array
correct = 0
incorrect = 0
p_errs = np.zeros(7)

# loops through each row in the true_vals array
for i in range(true_vals.shape[0]):
  # predicted value = predicted value from loop
  pred = y_pred[i]
  # true value = true value from loop
  true_val = true_vals[i]
  # Adds to error: absolute percent difference between true and predicted values
  p_errs += (np.abs((pred-true_val)/true_val))

# Outputs error percentages
print((p_errs/true_vals.shape[0])*100)

In [ ]:
# make a dataframe with the names and feat_dfs and inverse_feat_dfs:
real_imj_df = {"path":[],"feats_df":[],"inverse_feats_df":[]}

reg_paths = os.listdir("./content/ImageJ_data/Inverse_resized/")

for path in reg_paths:
  real_imj_df["path"].append(path)
  real_imj_df["feats_df"].append(pd.read_csv("./content/ImageJ_data/Regular_resized/" + path))
  real_imj_df["inverse_feats_df"].append(pd.read_csv("./content/ImageJ_data/Inverse_resized/" + path))

real_df = pd.DataFrame(real_imj_df)

In [ ]:
real_df["feats_df"][0]

In [ ]:
# Add regular features

# first list is mean, second list is std except for mean and median
feats = {'Mean':[],'Median':[],'Area':[[],[]], 'X':[[],[]], 'Y':[[],[]], 'Perim.':[[],[]], 'BX':[[],[]],
         'BY':[[],[]], 'Width':[[],[]], 'Height':[[],[]], 'Major':[[],[]], 'Minor':[[],[]],
       'Angle':[[],[]], 'Circ.':[[],[]], 'Feret':[[],[]], 'IntDen':[[],[]], '%Area':[[],[]],
       'RawIntDen':[[],[]], 'FeretX':[[],[]], 'FeretY':[[],[]], 'FeretAngle':[[],[]], 'MinFeret':[[],[]], 'AR':[[],[]],
       'Round':[[],[]], 'Solidity':[[],[]]}

# feats = {'Mean_inverted':[],'Median_inverted':[],'Area':[[],[]], 'X':[[],[]], 'Y':[[],[]], 'Perim.':[[],[]], 'BX':[[],[]],
#          'BY':[[],[]], 'Width':[[],[]], 'Height':[[],[]], 'Major':[[],[]], 'Minor':[[],[]],
#        'Angle':[[],[]], 'Circ.':[[],[]], 'Feret':[[],[]], 'IntDen':[[],[]], '%Area':[[],[]],
#        'RawIntDen':[[],[]], 'FeretX':[[],[]], 'FeretY':[[],[]], 'FeretAngle':[[],[]], 'MinFeret':[[],[]], 'AR':[[],[]],
#        'Round':[[],[]], 'Solidity':[[],[]]}


counter = 0

# iterate through all sims
for n in range(real_df.shape[0]): # grab the feats dataframe for this sim
  fdf = real_df["feats_df"][n] #iterate through all feats within the feats dictionary

  for i in range(len(feats.keys())): # select the current feat
    feat = list(feats.keys())[i]

    # if feat in ['Mean_inverted','Median_inverted']: # check for feats that should be added from the overall measurement (first row)
    if feat in ['Mean','Median']: # check for feats that should be added from the overall measurement (first row)
      # feats[feat].append(fdf[feat[0:-9]][0]) #inverse
      feats[feat].append(fdf[feat[:]][0])
      # feats[feat].append(fdf[feat][0])

    else:
      if fdf.shape[0] <= 1: # check if the fdf is only one measurement (empty result)
        feats[feat][0].append(np.mean(np.array(fdf[feat][0]))) # mean of sole measurement
        feats[feat][1].append(np.std(np.array(fdf[feat][0]))) #std of sole measurement (0)

      else:
        feats[feat][0].append(np.mean(np.array(fdf[feat][1:]))) # add mean for each of the rest of the feats using the remaining rows
        feats[feat][1].append(np.std(np.array(fdf[feat][1:]))) # add std for each of the rest of the feats using the remaining rows

  if counter % 1000 == 0:
    print(counter)
  counter += 1

print("Counting done")

In [ ]:
num_spots = []

for i in range(real_df.shape[0]):
  fdf = real_df["feats_df"][i] #iterate through all feats within the feats dictionary
  if fdf.shape[0] <= 1:
    num_spots.append(0)
  else:
    num_spots.append(fdf.shape[0]-1)

# feats_df["num_spots_inverted"] = num_spots
real_df["num_spots"] = num_spots

In [ ]:
for key in list(feats.keys()):
  # if key in ['Mean_inverted','Median_inverted']:
  if key in ['Mean','Median']:
    real_df[key] = feats[key]
  else:
    # feats_df[key + "_inverted_mean"] = feats[key][0]
    # feats_df[key + "_inverted_std"] = feats[key][1]
    real_df[key + "_mean"] = feats[key][0]
    real_df[key + "_std"] = feats[key][1]

In [ ]:
# Add regular features

# first list is mean, second list is std except for mean and median
# feats = {'Mean':[],'Median':[],'Area':[[],[]], 'X':[[],[]], 'Y':[[],[]], 'Perim.':[[],[]], 'BX':[[],[]],
#          'BY':[[],[]], 'Width':[[],[]], 'Height':[[],[]], 'Major':[[],[]], 'Minor':[[],[]],
#        'Angle':[[],[]], 'Circ.':[[],[]], 'Feret':[[],[]], 'IntDen':[[],[]], '%Area':[[],[]],
#        'RawIntDen':[[],[]], 'FeretX':[[],[]], 'FeretY':[[],[]], 'FeretAngle':[[],[]], 'MinFeret':[[],[]], 'AR':[[],[]],
#        'Round':[[],[]], 'Solidity':[[],[]]}

feats = {'Mean_inverted':[],'Median_inverted':[],'Area':[[],[]], 'X':[[],[]], 'Y':[[],[]], 'Perim.':[[],[]], 'BX':[[],[]],
         'BY':[[],[]], 'Width':[[],[]], 'Height':[[],[]], 'Major':[[],[]], 'Minor':[[],[]],
       'Angle':[[],[]], 'Circ.':[[],[]], 'Feret':[[],[]], 'IntDen':[[],[]], '%Area':[[],[]],
       'RawIntDen':[[],[]], 'FeretX':[[],[]], 'FeretY':[[],[]], 'FeretAngle':[[],[]], 'MinFeret':[[],[]], 'AR':[[],[]],
       'Round':[[],[]], 'Solidity':[[],[]]}


counter = 0

# iterate through all sims
for n in range(real_df.shape[0]): # grab the feats dataframe for this sim
  fdf = real_df["inverse_feats_df"][n] #iterate through all feats within the feats dictionary

  for i in range(len(feats.keys())): # select the current feat
    feat = list(feats.keys())[i]

    if feat in ['Mean_inverted','Median_inverted']: # check for feats that should be added from the overall measurement (first row)
    # if feat in ['Mean','Median']: # check for feats that should be added from the overall measurement (first row)
      feats[feat].append(fdf[feat[0:-9]][0]) #inverse
      # feats[feat].append(fdf[feat[:]][0])

    else:
      if fdf.shape[0] <= 1: # check if the fdf is only one measurement (empty result)
        feats[feat][0].append(np.mean(np.array(fdf[feat][0]))) # mean of sole measurement
        feats[feat][1].append(np.std(np.array(fdf[feat][0]))) #std of sole measurement (0)

      else:
        feats[feat][0].append(np.mean(np.array(fdf[feat][1:]))) # add mean for each of the rest of the feats using the remaining rows
        feats[feat][1].append(np.std(np.array(fdf[feat][1:]))) # add std for each of the rest of the feats using the remaining rows

  if counter % 1000 == 0:
    print(counter)
  counter += 1

print("Counter complete")

In [ ]:
num_spots = []

for i in range(real_df.shape[0]):
  fdf = real_df["inverse_feats_df"][i] #iterate through all feats within the feats dictionary
  if fdf.shape[0] <= 1:
    num_spots.append(0)
  else:
    num_spots.append(fdf.shape[0]-1)

real_df["num_spots_inverted"] = num_spots
# real_df["num_spots"] = num_spots

In [ ]:
for key in list(feats.keys()):
  if key in ['Mean_inverted','Median_inverted']:
  # if key in ['Mean','Median']:
    real_df[key] = feats[key]
  else:
    real_df[key + "_inverted_mean"] = feats[key][0]
    real_df[key + "_inverted_std"] = feats[key][1]
    # real_df[key + "_mean"] = feats[key][0]
    # real_df[key + "_std"] = feats[key][1]

In [ ]:
print(real_df.columns[3:101])

In [ ]:
# select features of real images
real_feats = real_df[real_df.columns[3:101]]

#scale data
scaling=StandardScaler()

# Use fit and transform method
scaling.fit(real_feats)
real_feats_scaled = scaling.transform(real_feats)



# predict params of images from real feats
y_pred = model.predict(real_feats_scaled)

real_df["pred_params"] = list(y_pred)

In [ ]:
real_df["pred_params"][0]

In [ ]:
real_df["pred_params"][0]

In [ ]:
for i in range(real_df.shape[0]):
  print(real_df["path"][i])
  print(real_df["pred_params"][i])